# Feature engineering
In this step we will introduce features learned from the data exploration step, re-formatting the information we have into a language the model can understand. <br>
This includes: <br>
- splitting wind speed into discrete bins to approach the "S" curve <br>
- splitting wind directions into discrete bins (NE, NW, SW, SE) <br>
- create cycling time feature (hour_of_day, month_of_year)


## Import necessary libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression

## Load data

In [ ]:
path1 = "../Data/Processed/wind_persistance_polar.csv"
path2 = "../Data/Processed/wind_average_polar.csv"
path3 = "../Data/Processed/wind_hourly_polar.csv"

def load_data(path):
    data = pd.read_csv(path)
    return data

wind_persistance_polar = load_data(path1)
wind_average_polar = load_data(path2)
wind_hourly_polar = load_data(path3)

wind_persistance_polar.info()

Somehow the timestamp is not datetime type 

## Create features

In [ ]:
def create_features(wind):
    """
    Create features to improve wind power forecasting.
    """
    wind_feat = wind.copy()
    wind_feat['TIMESTAMP'] = pd.to_datetime(wind_feat['TIMESTAMP']) 

    # 1. Wind speed bins using 100m (linearizing S-curve)
    # Based on the plots, we define three regions:
    # a. Cut-in: 0-4 m/s
    # b. Ramp-up: 4-20 m/s
    # c. Cut-out: >20 m/s

    speed_bins = [0, 4, 20, 50]
    speed_labels = ['cut-in', 'ramp-up', 'cut-out']

    wind_feat['wind_speed_bin'] = pd.cut(wind_feat['wind_speed_100m'], bins=speed_bins, labels=speed_labels, right=False, include_lowest=True)

    # 2. Wind direction bins
    dir_bins = [0, 90, 180, 270, 360]
    dir_labels = ['NE', 'SE', 'SW', 'NW']
    wind_feat['wind_dir_bin'] = pd.cut(wind_feat['wind_direction_100m'], bins=dir_bins, labels=dir_labels, right=False, include_lowest=True)

    # 3. Cyclical Time Features (hour)
    wind_feat['hour'] = wind_feat['TIMESTAMP'].dt.hour
    wind_feat['hour_sin'] = np.sin(2 * np.pi * wind_feat['hour'] / 24)
    wind_feat['hour_cos'] = np.cos(2 * np.pi * wind_feat['hour'] / 24)

    # 4. Cyclical Time Features (month)
    wind_feat['month'] = wind_feat['TIMESTAMP'].dt.month
    wind_feat['month_sin'] = np.sin(2 * np.pi * wind_feat['month'] / 12)
    wind_feat['month_cos'] = np.cos(2 * np.pi * wind_feat['month'] / 12)

    return wind_feat


### Apply features to all datasets

In [ ]:
# List of features to use
features_to_use = [
    'POWER',
    'wind_speed_100m',
    'wind_speed_bin',
    'wind_dir_bin',
    'hour_sin',
    'hour_cos',
    'month_sin',
    'month_cos'
]

# Create features for persistence data
wind_pers_feat = create_features(wind_persistance_polar)
wind_pers_feat = wind_pers_feat[features_to_use].copy()

# Create features for average data
wind_avg_feat = create_features(wind_average_polar)
wind_avg_feat = wind_avg_feat[features_to_use].copy()

# Create features for hourly data
wind_hour_feat = create_features(wind_hourly_polar)
wind_hour_feat = wind_hour_feat[features_to_use].copy()

print("\nDataFrames are ready for preprocessing.")

## Create preprocessing pipeline

In [ ]:
def get_preprocessor():
    """
    Define and return preprocessing pipeline.
    """
    # 1. Define feature lists
    numerical_features = ['wind_speed_100m']
    categorical_features = ['wind_speed_bin', 'wind_dir_bin']
    passthrough_features = ['hour_sin', 'hour_cos', 'month_sin', 'month_cos']

    # 2. Define the transformers
    numeric_transformer = Pipeline(steps=[
        ('scaler', StandardScaler())
    ])

    categorical_transformer = Pipeline(steps=[
        ('onehot', OneHotEncoder(handle_unknown='ignore'))
    ])

    # 3. Create the ColumnTransformer
    preprocessor = ColumnTransformer(
        transformers=[
            ('num', numeric_transformer, numerical_features),
            ('cat', categorical_transformer, categorical_features),
            ('pass', 'passthrough', passthrough_features)
        ]
    )

    return preprocessor


### Apply preprocessing to all datasets

Split feature data into X (input data) and y (output data). <br>
Fit the preprocessing pipeline to all three datasets.

In [ ]:
preprocessor = get_preprocessor()

# Apply preprocessing to persistance data
# 1. Split into X and y
X_pers = wind_pers_feat.drop('POWER', axis=1)
y_pers_train = wind_pers_feat['POWER']

# 2. Fit preprocessing on training data
X_pers_train = preprocessor.fit_transform(X_pers)
print("\nTransformed training data shape (persistence data):", X_pers_train.shape)

# Apply preprocessing for average data
# 1. Split into X and y
X_avg = wind_avg_feat.drop('POWER', axis=1)
y_avg_train = wind_avg_feat['POWER']

# 2. Fit preprocessing on training data
X_avg_train = preprocessor.fit_transform(X_avg)
print("\nTransformed training data shape (average data):", X_avg_train.shape)

# Apply preprocessing for hourly data
X_hour = wind_hour_feat.drop('POWER', axis=1)
y_hour_train = wind_hour_feat['POWER']

# 2. Fit preprocessing on training data
X_hour_train = preprocessor.fit_transform(X_hour)
print("\nTransformed training data shape (hourly data):", X_hour_train.shape)

### Machine learning model evaluation

As the wind speed and power output relationship is non-linear, we will compare the results of a Gradient Boosting model (e.g., XGBoost) with a Random Forest model. These will also be compared to a Linear Regression to further exemplify the non-linearity of the data set. <br>
We will evaluate the models performance using cross-validation and their results will be compared using MSE (Mean Square Error) and R² (Coefficient of Determination). <br>
The highest scoring model will be chosen for further fine tuning.

In [ ]:
from xgboost import XGBRegressor

# XGBoost with tuned parameters for wind power forecasting
xgb_model = XGBRegressor(
    n_estimators=1000,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    objective='reg:squarederror',
    verbosity=0,
    n_jobs=-1
)


In [ ]:
from sklearn.ensemble import RandomForestRegressor

rf_model = RandomForestRegressor(
    n_estimators=500,
    max_depth=20,
    min_samples_split=10,
    min_samples_leaf=5,
    random_state=42,
    n_jobs=-1
)


In [ ]:
from sklearn.linear_model import LinearRegression
linear_model = LinearRegression()


In [ ]:
from sklearn.model_selection import cross_validate
from sklearn.metrics import make_scorer, r2_score, mean_absolute_error
import numpy as np
import pandas as pd

#Results of all ML models for the Persistance Fill data set
models = {
    'Linear Regression': LinearRegression(),
    'Random Forest': RandomForestRegressor(n_estimators=100, random_state=42),
    'XGBoost': XGBRegressor(n_estimators=100, random_state=42)
}

scoring = {
    'r2': make_scorer(r2_score),
    'mae': make_scorer(mean_absolute_error, greater_is_better=False)
}

results = {}
for name, model in models.items():
    pipeline = Pipeline([('preprocessor', preprocessor), ('model', model)])
    # Use the raw DataFrame X_pers so ColumnTransformer can select columns by name
    cv_results = cross_validate(pipeline, X_pers, y_pers_train, cv=5, scoring=scoring, n_jobs=-1)
    results[name] = cv_results

# Summarize results into a small DataFrame and print
summary_list = []
for name, res in results.items():
    mean_r2 = np.mean(res['test_r2'])
    mean_mae = np.mean(res['test_mae'])
    # If MAE was returned as negative (scoring with greater_is_better=False), take absolute
    if mean_mae < 0:
        mean_mae = -mean_mae
    summary_list.append({'model': name, 'mean_r2': mean_r2, 'mean_mae': mean_mae})

print("Results for Persistance Fill")
summary_df = pd.DataFrame(summary_list).set_index('model').sort_values('mean_r2', ascending=False)
print(summary_df)


In [ ]:
#Results of all ML models for the Moving Average Fill data set
results = {}
for name, model in models.items():
    pipeline = Pipeline([('preprocessor', preprocessor), ('model', model)])
    # Use the raw DataFrame X_pers so ColumnTransformer can select columns by name
    cv_results = cross_validate(pipeline, X_avg, y_avg_train, cv=5, scoring=scoring, n_jobs=-1)
    results[name] = cv_results

# Summarize results into a small DataFrame and print
summary_list = []
for name, res in results.items():
    mean_r2 = np.mean(res['test_r2'])
    mean_mae = np.mean(res['test_mae'])
    # If MAE was returned as negative (scoring with greater_is_better=False), take absolute
    if mean_mae < 0:
        mean_mae = -mean_mae
    summary_list.append({'model': name, 'mean_r2': mean_r2, 'mean_mae': mean_mae})
print("Results for Moving Average Fill")
summary_df = pd.DataFrame(summary_list).set_index('model').sort_values('mean_r2', ascending=False)
print(summary_df)

In [ ]:
#Results of all ML models for the Hourly Climatology Fill data set
results = {}
for name, model in models.items():
    pipeline = Pipeline([('preprocessor', preprocessor), ('model', model)])
    # Use the raw DataFrame X_pers so ColumnTransformer can select columns by name
    cv_results = cross_validate(pipeline, X_hour, y_hour_train, cv=5, scoring=scoring, n_jobs=-1)
    results[name] = cv_results

# Summarize results into a small DataFrame and print
summary_list = []
for name, res in results.items():
    mean_r2 = np.mean(res['test_r2'])
    mean_mae = np.mean(res['test_mae'])
    # If MAE was returned as negative (scoring with greater_is_better=False), take absolute
    if mean_mae < 0:
        mean_mae = -mean_mae
    summary_list.append({'model': name, 'mean_r2': mean_r2, 'mean_mae': mean_mae})
print("Results for Hourly Climatology Fill")
summary_df = pd.DataFrame(summary_list).set_index('model').sort_values('mean_r2', ascending=False)
print(summary_df)

After cross-validation all models had quite high R2 and low MAE scores. However, XGBoost model preformed the best in both metrics, thus this model is chosen for further fine tuning. Additionally, the R2 score was the highest and MAE the lowest for the Persistance Fill set, so this method of filling in the missing power output values is chosen as the best option.

(Experiment with the rolling average some more)

### Fine Tuning

In [ ]:
# Fine-tune XGBoost hyperparameters with RandomizedSearchCV (maximize R2)
from sklearn.model_selection import RandomizedSearchCV, TimeSeriesSplit
from scipy.stats import randint, uniform
from xgboost import XGBRegressor
from sklearn.pipeline import Pipeline as SKPipeline
import pandas as pd

# Create a fresh XGBoost estimator inside a pipeline so params can be prefixed with 'model__'
xgb_base = XGBRegressor(objective='reg:squarederror', verbosity=0, n_jobs=-1, random_state=42)
pipeline_rs = SKPipeline([('preprocessor', preprocessor), ('model', xgb_base)])

# Parameter distributions (prefixed with 'model__' to target the estimator in the pipeline)
param_dist = {
    'model__n_estimators': randint(100, 1200),
    'model__max_depth': randint(3, 12),
    'model__learning_rate': uniform(0.01, 0.29),
    'model__subsample': uniform(loc=0.5, scale=0.5),
    'model__colsample_bytree': uniform(loc=0.5, scale=0.5),
    'model__gamma': uniform(loc=0, scale=5),
    'model__min_child_weight': randint(1, 10)
}

# TimeSeries CV to avoid leakage (use the same split count used earlier)
tscv_tune = TimeSeriesSplit(n_splits=5)

# Randomized search (optimize for R2). n_iter set modestly to keep runtime reasonable; increase if you have time.
rs = RandomizedSearchCV(
    estimator=pipeline_rs,
    param_distributions=param_dist,
    n_iter=40,
    scoring='r2',
    cv=tscv_tune,
    random_state=42,
    n_jobs=-1,
    verbose=2,
    return_train_score=False
)

# Fit the search on the persistence-filled dataset (this is the canonical dataset used earlier)
# NOTE: This can take time depending on n_iter and data size.

rs.fit(X_pers, y_pers_train)

# Save best estimator and results for later reuse
best_xgb = rs.best_estimator_
best_params = rs.best_params_
best_score = rs.best_score_
print('Best R2 (cv):', best_score)
print('Best params:')
for k, v in best_params.items():
    print(f'  {k}: {v}')



After fine tuning the XGBoost model using RandomizedSearchCV, the best parameters were found to be:
- n_estimators: 1061
- max_depth: 8
- learning_rate: 0.1884
- subsample: 0.6271
- colsample_bytree: 0.7850
- gamma: 0.4859
- min_child_weight: 7
The R2 score achieved with these parameters was 0.779, which is a small improvement from the default parameters.

In [ ]:
# Summarize top 5 results
cv_results = pd.DataFrame(rs.cv_results_)
top = cv_results.sort_values('mean_test_score', ascending=False).head(5)[[
    'mean_test_score', 'std_test_score', 'params'
]]
print(top)



### Evaluation of fine-tuned model on test set

Before evaluating the fine-tuned model on the test set, the test data must first be preprocessed using the same preprocessing pipeline as the training data. <br>
The fine-tuned model is then used to make predictions on the preprocessed test data. The performance of the model is evaluated using R².

In [ ]:
# Import processed weather data for test set evaluation
path4 = "../Data/Processed/weather_input_polar.csv"
# Importing the solution weather data to evaluate the fine-tuned model on the test set
path5 = "../Data/Solution_A.csv"

# Importing the Forecast template_A to create the submission file later
#path6 = "../Data/Processed/ForecastTemplate_A.csv"

def load_data(path):
    data = pd.read_csv(path)
    
    return data

weather_input_polar = load_data(path4)
solution_A_power = load_data(path5)

print(weather_input_polar.info())
print(weather_input_polar.head())
print("\n")
print(solution_A_power.info())
print(solution_A_power.head())

As the Weather Input data does not have the same features as the training data, we need to ensure that the preprocessing pipeline can handle this discrepancy.  <br>

Comment: I'm not sure this is a good way of actually implementing the final prediction on the weather input data, as it doesn't contain the POWER column, which is expected by the preprocessor. To remedy this I just added the missing column from the Solution_A.csv.

In [ ]:
# Create a new DataFrame `weather_input_polar_power` that includes the
# POWER column from `solution_A_power` placed immediately after TIMESTAMP.
import pandas as pd

# Work on a copy so we don't modify the original inputs in-place
weather_input_polar_power = weather_input_polar.copy()

# Ensure TIMESTAMP columns are comparable (convert in copies only)
weather_input_polar_power['TIMESTAMP'] = pd.to_datetime(weather_input_polar_power['TIMESTAMP'])
solution_power_for_merge = solution_A_power.copy()
solution_power_for_merge['TIMESTAMP'] = pd.to_datetime(solution_power_for_merge['TIMESTAMP'], format='%Y%m%d %H:%M', errors='coerce')

# Merge POWER into the weather data (left join to preserve weather rows).
weather_input_polar_power = weather_input_polar_power.merge(
    solution_power_for_merge[['TIMESTAMP', 'POWER']],
    on='TIMESTAMP',
    how='left'
)

# Reorder columns: place 'POWER' immediately after 'TIMESTAMP'
cols = weather_input_polar_power.columns.tolist()
if 'POWER' in cols:
    cols.remove('POWER')
    insert_idx = cols.index('TIMESTAMP') + 1 if 'TIMESTAMP' in cols else 1
    cols.insert(insert_idx, 'POWER')
    weather_input_polar_power = weather_input_polar_power[cols]

# Quick peek
weather_input_polar_power.head()


In [ ]:
# Prepare and run the weather input through the SAME preprocessing pipeline used for training
# 1) Create the same engineered features on the weather input DataFrame
weather_input_feat = create_features(weather_input_polar_power)
# Keep the same columns used for training (features_to_use defined earlier)
weather_input_feat = weather_input_feat[features_to_use].copy()

# 2) Split into X and y for the weather input (POWER may be present from the merge)
X_weather_input = weather_input_feat.drop('POWER', axis=1) if 'POWER' in weather_input_feat.columns else weather_input_feat.copy()
y_weather_input = weather_input_feat['POWER'] if 'POWER' in weather_input_feat.columns else None

# 3) Fit the preprocessor on the persistence training data (fit once), then transform both sets
# This is a check, so I can be sure the preprocessor results are consistent with earlier steps
# Can delete this step later
preprocessor = get_preprocessor()
preprocessor.fit(X_pers)  # X_pers was created earlier from wind_pers_feat
X_pers_train = preprocessor.transform(X_pers)
print("\nTransformed training data shape (persistence data):", X_pers_train.shape)

# 4) Transform weather input using the fitted preprocessor
X_weather_input_transformed = preprocessor.transform(X_weather_input)
print("\nTransformed weather input data shape (weather input):", X_weather_input_transformed.shape)

In [ ]:
#Evaluate the final model on the weather input data set using R2 score
weather_input_test = rs.best_estimator_

X_weather_input = weather_input_feat.drop('POWER', axis=1)
y_weather_input = weather_input_feat['POWER'].copy()
y_weather_input = weather_input_feat['POWER']

# Predict on the power data
X_weather_input_transformed = preprocessor.transform(X_weather_input)
predictions = weather_input_test.predict(X_weather_input_transformed)

final_r2 = r2_score(y_weather_input, predictions)
print("Final R2 on weather input test set:", final_r2)
